# 02 · LLM Fundamentals

> **Source notes:** `LLMFundamentals.md`

What actually happens inside a language model? This notebook makes it concrete:
- **Tokenisation** — BPE in action with `tiktoken`
- **Sampling parameters** — temperature, top-p, top-k via a local Ollama SLM
- **Lost-in-the-middle** — empirical context position experiment
- **Training stages mental model** — prompt-level SFT vs RLHF behaviour differences

All LLM calls use **Ollama** (local, no API key needed).

## 0 · Environment Setup

```bash
# One-time: install Ollama and pull a model
winget install Ollama.Ollama        # Windows
# brew install ollama               # macOS
ollama pull phi3:mini               # ~2 GB download
```
Make sure `ollama serve` is running (or the desktop app is open) before executing cells.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "tiktoken", "ollama", "-q"], check=True)
print("Packages ready.")

## 1 · Tokenisation — BPE in Action

The model never sees raw text. Text is broken into **tokens** (subword units) by a byte-pair encoding (BPE) vocabulary.

Key facts:
- ~1 token ≈ 0.75 English words
- Numbers tokenise byte-by-byte in some vocabs → arithmetic is hard
- The same string tokenises differently across model families

`tiktoken` is OpenAI's tokeniser library. `cl100k_base` is the vocabulary used by GPT-4 / GPT-3.5.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / GPT-3.5 vocabulary

samples = [
    "The transformer architecture changed everything.",
    "self.attention_weights[layer_idx]",          # code is token-dense
    "1234567890",                                 # numbers tokenise oddly
    "tokenisation",                               # British spelling → split
    "Mamma Rosa's cheapest gluten-free pizza",    # running example
]

print(f"{'Text':<45} {'Tokens':>6}  Token IDs")
print("-" * 90)
for text in samples:
    token_ids = enc.encode(text)
    tokens    = [enc.decode([t]) for t in token_ids]
    print(f"{text!r:<45} {len(token_ids):>6}  {tokens}")

In [ ]:
# Cost estimation helper — common real-world use

def estimate_cost(text: str, model_tier: str = "mid") -> dict:
    """Estimate token count and rough cost for an input string."""
    RATES = {
        "frontier": {"input": 5.00,  "output": 15.00},  # GPT-4o class  $/1M tokens
        "mid":      {"input": 0.15,  "output": 0.60},   # GPT-4o-mini class
        "embed":    {"input": 0.02,  "output": 0.0},    # text-embedding-3-small
    }
    n_tokens = len(enc.encode(text))
    rate     = RATES[model_tier]
    cost_usd = n_tokens / 1_000_000 * rate["input"]
    return {"tokens": n_tokens, "model_tier": model_tier,
            "input_cost_usd": round(cost_usd, 8)}

# Simulate a RAG prompt: system + retrieved chunks + user query
system_prompt    = "You are a helpful assistant for Mamma Rosa's pizza ordering system. Answer only using the provided menu context."
retrieved_chunks = "Menu: Margherita £9.99 (gf option +£1.50), Pepperoni £11.99, Garlic Bread £3.49. Delivery zones: central, north. Min order £15."
user_query       = "What's the cheapest gluten-free pizza available for delivery?"
full_prompt      = f"{system_prompt}\n\nContext:\n{retrieved_chunks}\n\nUser: {user_query}"

result = estimate_cost(full_prompt, model_tier="mid")
print(f"Full RAG prompt: {result['tokens']} tokens  →  ${result['input_cost_usd']:.6f} USD (input, mid-tier)")
print(f"At 10,000 req/day: ${result['input_cost_usd'] * 10_000:.2f} USD/day input cost")

## 2 · Sampling Parameters — Temperature, Top-p, Top-k

The model outputs a probability distribution over vocabulary at each step. Sampling parameters control how you draw the next token.

| Parameter | Effect |
|---|---|
| **Temperature = 0** | Argmax — always pick the most probable token. Deterministic, factual. |
| **Temperature = 1** | Sample from raw distribution. Creative, varied. |
| **Temperature > 1** | Flatten distribution → more random/chaotic. |
| **Top-p (nucleus)** | Only sample from smallest set of tokens summing to probability ≥ p. |
| **Top-k** | Only sample from the k most probable tokens. |

Below we ask the same factual question at different temperatures and observe consistency.

In [ ]:
import ollama

MODEL = "phi3:mini"  # change to any model you have pulled

def ask(prompt: str, temperature: float, top_p: float = 0.9, n: int = 3) -> list[str]:
    """Ask the same question n times at a given temperature and return all responses."""
    responses = []
    for _ in range(n):
        resp = ollama.chat(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": temperature, "top_p": top_p, "num_predict": 40}
        )
        responses.append(resp["message"]["content"].strip())
    return responses

prompt = "In one sentence: what is the capital of France?"

for temp in [0.0, 0.7, 1.5]:
    print(f"\n── Temperature = {temp} ─────────────────────")
    for i, ans in enumerate(ask(prompt, temperature=temp), 1):
        print(f"  [{i}] {ans[:120]}")

## 3 · Lost-in-the-Middle — Context Position Matters

Models recall information from the **start** and **end** of long contexts more reliably than from the **middle**. This is the *lost-in-the-middle* effect (Liu et al., 2023).

We simulate it by embedding a target fact at different positions in a padded context and measuring retrieval accuracy.

In [ ]:
import ollama

TARGET_FACT  = "The secret discount code for large pizzas is ROSE42."
PADDING_SENT = "Mamma Rosa's was founded in 1987 by Rosa Conti in Naples before expanding across Europe. "
PAD_N        = 15   # sentences of padding on each side
QUERY        = "What is the secret discount code for large pizzas? Reply with ONLY the code."

def build_context(position: str) -> str:
    pad = PADDING_SENT * PAD_N
    if position == "start":
        return TARGET_FACT + " " + pad
    elif position == "middle":
        half = PAD_N // 2
        return PADDING_SENT * half + TARGET_FACT + " " + PADDING_SENT * half
    else:  # end
        return pad + TARGET_FACT

results = {}
for pos in ["start", "middle", "end"]:
    context = build_context(pos)
    prompt  = f"Context:\n{context}\n\nQuestion: {QUERY}"
    resp    = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 20}
    )
    answer = resp["message"]["content"].strip()
    found  = "ROSE42" in answer
    results[pos] = {"answer": answer, "found": found}
    print(f"Position={pos:<6}  Found={'✓' if found else '✗'}  Answer: {answer[:60]}")

print("\nConclusion: start and end positions retrieve reliably; middle may fail.")

## 4 · Training Stages Mental Model — SFT vs RLHF Behaviour

Three stages produce the assistant you call via API:

```
Stage 1: Pretraining   Raw transformer → learns language + world knowledge
Stage 2: SFT           Fine-tuned on (instruction, good response) pairs → follows instructions
Stage 3: RLHF / DPO   Aligned with human preferences → helpful, harmless, honest
```

We can probe the effects at the prompt level: a base-style completion prompt vs an instruction-following prompt shows how strongly the RLHF alignment suppresses raw continuation.

In [ ]:
import ollama

# Instruction-following (post-SFT/RLHF) — the model respects constraints
instruction_prompt = (
    "You are a concise assistant. Answer in exactly ONE word.\n"
    "Question: What is 2 + 2?"
)

# Raw completion style (no instruction wrapper) — model may be more verbose
completion_prompt = "2 + 2 ="

for label, prompt in [("Instruction-style", instruction_prompt),
                      ("Completion-style",  completion_prompt)]:
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 30}
    )
    print(f"\n── {label} ─────────────────────")
    print(f"  Prompt:   {prompt!r}")
    print(f"  Response: {resp['message']['content'].strip()!r}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Tokenisation | ~1 token ≈ 0.75 words; numbers and code are denser; use `tiktoken` for cost estimation |
| Temperature | 0 = deterministic factual; 0.7 = creative; >1 = noisy. Match to task. |
| Top-p / Top-k | Nucleus sampling limits the candidate pool; use with moderate temperature |
| Context position | Facts at start/end recalled better than middle — put critical context first |
| Training stages | Pretraining → SFT → RLHF produce progressively more useful and safe assistants |

**Next:** [PromptEngineering/notebook.ipynb](../PromptEngineering/notebook.ipynb) — build on this with reliable prompt patterns.